# Blackjack

## The Deck

Blackjack deck has 52 cards:
Ace, Number cards with 2 to 10 and face cards jack, queen and king, in the colors Spades (♠), Hearts (♥), Diamonds (♦), Clubs (♣)

Cards have different values.
Number cards have the value they show, fave cards have a value of 10, ace has 11 (or 1, in specific cases which will be dealt with later).

## The Rules

1. Each player and the dealer get two cards initially.
2. The player can choose to "play" (take another card) or "stop" (hold the current score).
3. Exceeding 21 points results in an automatic loss.
4. The winner is the player with the highest score ≤ 21 after comparing with the dealer. 

# Experiment



Imports & initialise

In [17]:
import numpy as np
import collections
# Hier nimmst du an, dass deine BlackjackEnv-Klasse im selben Verzeichnis liegt oder oben definiert wurde
# env = BlackjackEnv()

# Hyperparameter
alpha = 0.01
gamma = 1.0
epsilon = 0.1  # Exploration (Zufallszüge)
num_episodes = 500000

# Q-Tabelle initialisieren (Standardmäßig alles mit 0.0 besetzen)
# Ein defaultdict hilft uns, Zustände dynamisch zu erzeugen, wenn der Agent sie sieht
Q = collections.defaultdict(lambda: np.zeros(2))  # 0: Stand, 1: Hit

def choose_action(state, epsilon):
    """Epsilon-Greedy Strategie für die Aktionsauswahl"""
    if random.random() < epsilon:
        return random.choice([0, 1])  # Erkundung
    else:
        return np.argmax(Q[state])  # Ausbeutung des Wissens

Training Loop


In [ ]:
env = BlackjackEnv()
win_loss_draw = [0, 0, 0] 

for episode in range(num_episodes):
    state = env.reset()
    done = False
    
    while not done:
        action = choose_action(state, epsilon)
        
        # Aktion ausführen
        if action == 1:
            next_state, reward, done = env.hit()
        else:
            next_state, reward, done = env.stand()
            
        # Q-Value Update
        best_next_action = np.argmax(Q[next_state])
        td_target = reward + gamma * Q[next_state][best_next_action] * (not done)
        td_delta = td_target - Q[state][action]
        Q[state][action] += alpha * td_delta
        
        state = next_state
        
    # Statistik am Ende der Runde erfassen
    if reward == 1.0:
        win_loss_draw[0] += 1
    elif reward == -1.0:
        win_loss_draw[1] += 1
    else:
        win_loss_draw[2] += 1

print(f"Training beendet nach {num_episodes} Episoden.")
print(f"Gewonnen: {win_loss_draw[0]} | Verloren: {win_loss_draw[1]} | Unentschieden: {win_loss_draw[2]}")

Testing the learned strategy (without randomness)

In [ ]:
# Testphase: Epsilon auf 0 setzen, um nur noch die beste Strategie zu spielen
test_episodes = 10000
test_wins = 0

for _ in range(test_episodes):
    state = env.reset()
    done = False
    while not done:
        action = np.argmax(Q[state])  # Strikt nach Q-Tabelle handeln
        if action == 1:
            _, reward, done = env.hit()
        else:
            _, reward, done = env.stand()
    if reward == 1.0:
        test_wins += 1

print(f"Win-Rate des Agenten nach dem Training: {test_wins / test_episodes * 100:.2f}%")